# Cleaning dirty and messy data with python

In [1]:
import pandas as pd
import numpy as np

# Dataset context

- The sensor location indicates the location where the sensor is placed and the system (A, B) it corresponds to (e.g. Entry A).
- The dates indicates the days where data was collected from the sensors (e.g. 1/5/2023).
- The numeric values indicate the pressure sensors' measurements (ranging from 0-25 bar)

In [3]:
df = pd.read_csv("Downloads/pressure_sensor_data.tsv", sep="\t")
df.head()

,Sensor Location,1/5/2023,1/6/2023,1/18/2023,1/28/2023,2/6/2023,2/7/2023,2/9/2023,2/28/2023
0,Entry A,29.0,37.0,NaN,NaN,23.0,16.0,13.0,NaN
1,Area A,0.0,11.0,14.0,2.0,11.0,4.0,15.0,15.0
2,Area B,NaN,28.0,0.0,21.0,19.0,23.0,6.0,16.0
3,Entry A,6.0,NaN,4.0,5.0,NaN,NaN,11.0,3.0
4,Exit A,33.0,7.0,2.0,22.0,23.0,11.0,NaN,13.0


# 1.Tidiness Issue: Column names are values, not variable names.

In [6]:
updated_df = pd.melt(df, id_vars="Sensor Location", var_name="Date", value_name="Pressure")
updated_df.head()

,Sensor Location,Date,Pressure
0,Entry A,1/5/2023,29.0
1,Area A,1/5/2023,0.0
2,Area B,1/5/2023,NaN
3,Entry A,1/5/2023,6.0
4,Exit A,1/5/2023,33.0


# 2.Tidiness Issue: Multiple variables stored in one column

In [8]:
updated_df[["Sensor Location", "System"]] = updated_df["Sensor Location"].str.split(expand=True)

In [11]:
updated_df.head()

,Sensor Location,Date,Pressure,System
0,Entry,1/5/2023,29.0,A
1,Area,1/5/2023,0.0,A
2,Area,1/5/2023,NaN,B
3,Entry,1/5/2023,6.0,A
4,Exit,1/5/2023,33.0,A


# Quality issue: Validity and Accuracy

 Between Jan 5th to Jan 6th, some of sensors' data for both Systems A and B were found to be corruputed due to a sudden, very high increase in temperature.

Filter out rows where the sensors' reported data is greater than 25 bar into a seperate dataframe, and drop these rows from the original dataframe.

In [13]:
invalid_data = updated_df[(updated_df["Pressure"] > 25)]
invalid_data.index

Index([0, 4, 6, 8], dtype='int64')

In [14]:
#Dropping the invalid rows 
updated_df = updated_df.drop(invalid_data.index)
#Reset the index 
updated_df = updated_df.reset_index(drop=True)

updated_df.head()

,Sensor Location,Date,Pressure,System
0,Area,1/5/2023,0.0,A
1,Area,1/5/2023,NaN,B
2,Entry,1/5/2023,6.0,A
3,Exit,1/5/2023,18.0,B
4,Area,1/6/2023,11.0,A


In [15]:
updated_df["Pressure"].describe()

count    36.000000
mean     11.583333
std       6.983143
min       0.000000
25%       5.000000
50%      12.000000
75%      16.000000
max      23.000000
Name: Pressure, dtype: float64

The pressure variable range is correct in between 0-25bar.